In [ ]:
import copy
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F

from sklearn.model_selection import train_test_split

from torch_geometric.loader import DataLoader
from torch_geometric.nn import GINConv, global_mean_pool, BatchNorm

from preprocessing import create_dataset

In [ ]:
df = pd.read_excel("data-out.xlsx")
print(df.shape)

#df = df[df["Thermal_cond"] <= 0.4].reset_index(drop=True)
#print(df.shape)

dataset = create_dataset(df)

print(len(dataset))


train_dataset, valid_dataset = train_test_split(dataset, test_size=0.2, random_state=42, shuffle=True)

print(len(train_dataset))
print(len(valid_dataset))


train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

valid_loader = DataLoader(valid_dataset, batch_size=32, shuffle=False)

(1012, 19)
(1008, 19)
1008
806
202


In [5]:
class GIN(nn.Module):

    def __init__(self, num_node_features, hidden_dim=128, dropout=0.2):

        super().__init__()

        self.conv1 = GINConv(

            nn.Sequential(

                nn.Linear(
                    num_node_features,
                    hidden_dim
                ),

                nn.ReLU(),

                nn.Linear(
                    hidden_dim,
                    hidden_dim
                )

            )

        )

        self.bn1 = BatchNorm(hidden_dim)

        self.conv2 = GINConv(

            nn.Sequential(

                nn.Linear(
                    hidden_dim,
                    hidden_dim
                ),

                nn.ReLU(),

                nn.Linear(
                    hidden_dim,
                    hidden_dim
                )

            )

        )

        self.bn2 = BatchNorm(hidden_dim)

        self.conv3 = GINConv(

            nn.Sequential(

                nn.Linear(
                    hidden_dim,
                    hidden_dim
                ),

                nn.ReLU(),

                nn.Linear(
                    hidden_dim,
                    hidden_dim
                )

            )

        )

        self.bn3 = BatchNorm(hidden_dim)

        self.dropout = nn.Dropout(dropout)

        self.regressor = nn.Linear(hidden_dim, 1)

    def forward(self, data):

        x = data.x

        edge_index = data.edge_index

        batch = data.batch


        x = self.conv1(x, edge_index)

        x = self.bn1(x)

        x = F.relu(x)


        x = self.conv2(x, edge_index)

        x = self.bn2(x)

        x = F.relu(x)


        x = self.conv3(x, edge_index)

        x = self.bn3(x)

        x = F.relu(x)


        x = global_mean_pool(x, batch)

        embedding = self.dropout(x)

        prediction = self.regressor(embedding)

        return prediction.squeeze(-1), embedding

In [6]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = GIN(num_node_features=dataset[0].x.shape[1],hidden_dim=128,dropout=0.2).to(device)

print(model)

GIN(
  (conv1): GINConv(nn=Sequential(
    (0): Linear(in_features=6, out_features=128, bias=True)
    (1): ReLU()
    (2): Linear(in_features=128, out_features=128, bias=True)
  ))
  (bn1): BatchNorm(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (conv2): GINConv(nn=Sequential(
    (0): Linear(in_features=128, out_features=128, bias=True)
    (1): ReLU()
    (2): Linear(in_features=128, out_features=128, bias=True)
  ))
  (bn2): BatchNorm(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (conv3): GINConv(nn=Sequential(
    (0): Linear(in_features=128, out_features=128, bias=True)
    (1): ReLU()
    (2): Linear(in_features=128, out_features=128, bias=True)
  ))
  (bn3): BatchNorm(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (dropout): Dropout(p=0.2, inplace=False)
  (regressor): Linear(in_features=128, out_features=1, bias=True)
)


In [7]:
criterion = nn.MSELoss()

optimizer = torch.optim.AdamW( model.parameters(), lr=1e-3, weight_decay=1e-4)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=8)

In [8]:

def train_one_epoch(model, loader, optimizer, criterion, device):

    model.train()

    total_loss = 0

    for batch in loader:

        batch = batch.to(device)

        optimizer.zero_grad()

        prediction, _ = model(batch)

        loss = criterion(prediction, batch.y.view(-1))

        loss.backward()

        optimizer.step()

        total_loss += loss.item() * batch.num_graphs

    return total_loss / len(loader.dataset)

In [9]:
@torch.no_grad()

def evaluate(model, loader, criterion, device):
    
    model.eval()

    total_loss = 0

    predictions = []

    targets = []

    for batch in loader:

        batch = batch.to(device)

        prediction, _ = model(batch)

        loss = criterion( prediction, batch.y.view(-1) )
            

        total_loss += loss.item() * batch.num_graphs

        predictions.extend(prediction.cpu().numpy())

        targets.extend(batch.y.view(-1).cpu().numpy())

    loss = total_loss / len(loader.dataset)

    return (loss, np.array(predictions), np.array(targets))

In [10]:
best_loss = np.inf
best_state = None
patience = 20
counter = 0
epochs = 250

In [11]:
for epoch in range(1, epochs + 1):

    train_loss = train_one_epoch(model, train_loader, optimizer, criterion, device)

    valid_loss, pred, true = evaluate(model, valid_loader, criterion, device)

    scheduler.step(valid_loss)

    print(

        f"Epoch {epoch:03d}"

        f" | Train {train_loss:.5f}"

        f" | Valid {valid_loss:.5f}"

    )

    if valid_loss < best_loss:
        best_loss = valid_loss
        best_state = copy.deepcopy(model.state_dict())
        counter = 0

    else:
        counter += 1

    if counter >= patience:
        print("\nEarly Stopping")
        break


Epoch 001 | Train 0.08694 | Valid 0.09474
Epoch 002 | Train 0.04279 | Valid 0.02373
Epoch 003 | Train 0.03585 | Valid 0.00561
Epoch 004 | Train 0.03087 | Valid 0.01229
Epoch 005 | Train 0.02901 | Valid 0.00819
Epoch 006 | Train 0.01673 | Valid 0.00198
Epoch 007 | Train 0.01584 | Valid 0.00348
Epoch 008 | Train 0.01299 | Valid 0.00297
Epoch 009 | Train 0.01286 | Valid 0.00299
Epoch 010 | Train 0.01143 | Valid 0.00232
Epoch 011 | Train 0.01009 | Valid 0.00272
Epoch 012 | Train 0.00812 | Valid 0.00167
Epoch 013 | Train 0.00889 | Valid 0.00178
Epoch 014 | Train 0.00879 | Valid 0.00219
Epoch 015 | Train 0.00777 | Valid 0.00123
Epoch 016 | Train 0.00731 | Valid 0.00184
Epoch 017 | Train 0.00711 | Valid 0.00166
Epoch 018 | Train 0.00701 | Valid 0.00110
Epoch 019 | Train 0.00437 | Valid 0.00102
Epoch 020 | Train 0.00528 | Valid 0.00205
Epoch 021 | Train 0.00418 | Valid 0.00106
Epoch 022 | Train 0.00366 | Valid 0.00085
Epoch 023 | Train 0.00441 | Valid 0.00082
Epoch 024 | Train 0.00327 | Valid 

In [12]:
torch.save(best_state,"best_model.pt")

print("Model Saved")

model.load_state_dict(torch.load("best_model.pt",map_location=device))

model.eval()

full_loader = DataLoader(dataset,batch_size=32,shuffle=False)

Model Saved


In [13]:
embeddings = []

targets = []

model.eval()

with torch.no_grad():

    for batch in full_loader:

        batch = batch.to(device)

        _, embedding = model(batch)

        embeddings.append(embedding.cpu().numpy())

        targets.extend(batch.y.view(-1).cpu().numpy())

In [14]:

embeddings = np.concatenate(embeddings, axis=0)

print(embeddings.shape)

(1008, 128)


In [15]:
df_embeddings = pd.DataFrame(embeddings,columns=[f"GNN_{i}" for i in range(embeddings.shape[1])])

df_embeddings.to_csv("embedding.csv",index=False)

print("Embedding Saved.")

Embedding Saved.
